# The strawberry field is worth $3,847. The manure is worth $25,000.

Kaggriculture publishes its entire economy in `README.md`: seed costs, yield rates, and —
crucially — the exact price function the market uses. That means the profitable strategies
are *computable* before you write a single line of agent code. This notebook computes them.

Three results, all falling straight out of the published numbers:

- **Base prices are nearly meaningless.** A full 5×5 field of strawberries produces about
  100 units. Selling 100 strawberries earns **$3,847**, not the $12,000 the $120 base price
  suggests, because the 62nd one already sells for a dollar.
- **Melon is still the best crop**, but by a much smaller margin than the table implies:
  $26,627 for a field, not $75,000.
- **Fertilizer — a free byproduct of owning animals, produced whether or not you feed
  them — has the second-highest revenue ceiling in the game.** For most animals it is worth
  more than the animal's actual product.

Everything below is derived from the published rules. I have not run it against the live
environment, so treat it as a hypothesis with the arithmetic already done — and please
correct me in the comments if the engine disagrees.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, GREY = '#3987e5', '#d95926', '#8a8a85'
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#cccccc', 'font.size': 11,
                     'axes.titlesize': 14, 'axes.titleweight': 'bold',
                     'axes.titlelocation': 'left', 'axes.titlepad': 14})

# price(inv) = base + sign · amp · f(|inv − I0|),  amp = target · base / f(T)
SHAPE = {'linear': lambda x: x, 'sq': lambda x: x ** 2,
         'sqrt': lambda x: np.sqrt(x), 'log': lambda x: np.log1p(x)}

# base, T (one field's 24-day output), glut shape, glut target   — README "Price Function"
MARKET = {
    'Wheat':      (25, 400, 'log',    0.20),
    'Carrot':     (35, 450, 'sqrt',   0.70),
    'Tomato':     (60, 200, 'sqrt',   0.60),
    'Strawberry': (120, 100, 'linear', 1.60),
    'Melon':      (250, 300, 'sq',     3.60),
    'Egg':        (50, 332, 'log',    0.20),
    'Milk':       (160, 122, 'linear', 1.60),
    'Wool':       (200, 105, 'sq',     3.20),
    'Fertilizer': (100, 200, 'linear', 0.40),
}

def price(res, k):
    """Price of the (k+1)-th unit you sell, starting from an untouched market."""
    base, T, f, target = MARKET[res]
    amp = target * base / SHAPE[f](T)
    return np.maximum(np.round(base - amp * SHAPE[f](np.asarray(k, float))), 1)

def revenue(res, n):
    return float(price(res, np.arange(int(n))).sum())

# sanity check against the table printed in the README
chk = pd.DataFrame({r: [price(r, T), price(r, 2 * T)]
                    for r, (_, T, _, _) in MARKET.items()},
                   index=['P(I0+T)', 'P(I0+2T)']).T.astype(int)
chk['README P(I0+T)'] = [20, 10, 24, 1, 1, 40, 1, 1, 60]
chk['README P(I0+2T)'] = [19, 1, 9, 1, 1, 39, 1, 1, 20]
print(chk.to_string())
print('\nreproduces the README table exactly:',
      bool((chk['P(I0+T)'] == chk['README P(I0+T)']).all() and
           (chk['P(I0+2T)'] == chk['README P(I0+2T)']).all()))

The implementation matches every published price point, so the curve below is the
real one, not a guess.

## 1. What your own selling does to the price

Sell orders are processed one unit at a time and each unit you add to market inventory
pushes the price down for the next. The shape of that decay is chosen per resource, and
the README's own words are that premium goods "drive straight to the $1 floor".

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for r in MARKET:
    k = np.arange(0, 600)
    prem = MARKET[r][0] > 100
    ax = axes[0] if prem else axes[1]
    ax.plot(k, price(r, k), lw=2, label=f'{r}  (base ${MARKET[r][0]})')
for ax, ttl in zip(axes, ['Premium goods: base > $100', 'Staples: base ≤ $100']):
    ax.set_xlabel('units you have sold'); ax.set_ylabel('price of the next unit')
    ax.legend(frameon=False, fontsize=10)
    ax.axhline(1, color=GREY, lw=1, ls=':')
    ax.set_title(ttl)
plt.tight_layout(); plt.show()

floor = {}
for r in MARKET:
    p = price(r, np.arange(4000))
    floor[r] = int(np.argmax(p <= 1)) if (p <= 1).any() else None
print('units you can sell before the price is $1:')
for r, v in floor.items():
    print(f'  {r:11s} {"never" if v is None else v}')

Strawberry hits the floor after **62 units**. Wool after 59, milk after 76, melon
after 158. Wheat and egg never do — their glut curves are logarithmic, so they settle
around $19 and $39 and stay there no matter how much you dump.

This is the single most important table in the game and it is not the one in the README.

## 2. What a field is actually worth

`T` in the price table is defined as the output of one 5×5 field over 24 days at optimal
watering. So `revenue(T)` answers directly: **if I dedicate a quadrant to this crop for the
season and sell everything, what do I get?**

In [ ]:
rows = []
for r, (base, T, _, _) in MARKET.items():
    got = revenue(r, T)
    rows.append({'product': r, 'base': base, 'field output T': T,
                 'naive T×base': T * base, 'actual revenue': round(got),
                 'glut tax': f'{1 - got / (T * base):.0%}',
                 '$/unit': round(got / T, 1)})
tbl = pd.DataFrame(rows).sort_values('actual revenue', ascending=False)
print(tbl.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5.4))
t = tbl.iloc[::-1]
y = np.arange(len(t))
ax.barh(y, t['naive T×base'], .62, color='#dfe6ee', label='if base price held')
ax.barh(y, t['actual revenue'], .62, color=BLUE, label='what you actually get')
for i, (a, b) in enumerate(zip(t['actual revenue'], t['naive T×base'])):
    ax.text(b + 900, i, f'−{1 - a / b:.0%}', va='center', fontsize=10, color=ORANGE)
ax.set_yticks(y, t['product'])
ax.set_xlabel('revenue from one field-season, $')
ax.set_xlim(0, tbl['naive T×base'].max() * 1.16)
ax.tick_params(left=False)
ax.legend(frameon=False, loc='lower right')
ax.set_title('One 5×5 field, fully sold')
plt.show()

Sorted by what you actually bank rather than by base price, the ranking changes
completely:

Melon ($26,627), fertilizer ($16,020) and egg ($13,839) take the top three. Wool, milk and
strawberry — the three most expensive-looking goods on the price list — finish 6th, 8th
and 9th.

Strawberry has the second-highest base price in the game and the **lowest** field revenue
of any product. Its glut curve is linear with a target of 1.6, which means 100 units —
exactly one field's output — moves the price by 1.6× the base, i.e. straight through zero
to the floor. The card is designed to punish anyone who reads only the yield table.

Melon survives its 64% glut tax and still wins, because $250 is a long way to fall.

## 3. The manure economy

Now the part that is easy to miss. From the action list:

> **COLLECT_FERTILIZER** — *Every surviving animal makes 1 available at the end of each
> day, **whether or not it was fed or cared for**.*

Fertilizer is not a product you farm. It is a per-animal-per-day dividend that costs one
action to collect and nothing to produce. And its glut curve is the gentlest of any
premium-priced good: linear, target 0.40, so 400 units still average $60.

In [ ]:
for n in (100, 200, 300, 400, 500):
    print(f'{n:>4} fertilizer -> ${revenue("Fertilizer", n):>8,.0f}   '
          f'avg ${revenue("Fertilizer", n) / n:5.1f}/unit')

print('\nfor comparison, the animals\' own products:')
for r in ('Egg', 'Milk', 'Wool'):
    for n in (50, 100, 200):
        print(f'  {r:5s} {n:>4} -> ${revenue(r, n):>8,.0f}   '
              f'avg ${revenue(r, n) / n:5.1f}')

In [ ]:
DAYS = 20          # a realistic productive window inside the 30-day season
ANIMAL = {'Goose': ('Egg', 1, 300), 'Cow': ('Milk', 2, 400), 'Sheep': ('Wool', 3, 500)}

rows = []
for a, (prod, interval, cost) in ANIMAL.items():
    n_prod = DAYS // interval
    # value each stream at its average price over a plausible season volume
    unit_prod = revenue(prod, 100) / 100
    unit_fert = revenue('Fertilizer', 300) / 300
    rows.append({'animal': a, 'buy cost': cost, 'product': prod,
                 f'units in {DAYS}d': n_prod,
                 'product $': round(n_prod * unit_prod),
                 'fertilizer units': DAYS,
                 'fertilizer $': round(DAYS * unit_fert),
                 'manure share': f'{DAYS * unit_fert / (DAYS * unit_fert + n_prod * unit_prod):.0%}'})
print(pd.DataFrame(rows).to_string(index=False))

Per animal, over twenty days, valued at realistic season volumes, **the manure is
worth more than the animal's actual product in all three cases** — 62% of a goose's
income, 69% of a cow's, 75% of a sheep's.

And it is the *robust* half of that income: it needs no wheat, no feeding schedule, and it
keeps coming while the animal is neglected. A sheep is not a wool machine that also
fertilises. It is a fertiliser machine that occasionally produces wool.

Two consequences worth testing in an agent:

1. **Feeding is only required every other day** — "they will escape if not fed for *two
   successive* days" — and an unfed animal still produces its base yield, losing only the
   banked CARE bonus. Feeding daily doubles the action cost of an animal for a bonus that
   is capped by `max_held` anyway.
2. Fertilizer is also *usable*: it doubles a crop's per-day yield bonus for three days. So
   every unit is a choice between roughly $70 of sale value and a yield multiplier on a
   melon. Given melon is the only crop that survives its own glut, that trade is probably
   worth making — for melons, and almost nothing else.

## 4. The budget nobody mentions: 720 actions

There are 24 turns per day and 30 days: **720 farmer actions for the whole season**, plus
whatever you buy with `HIRE` at a Fibonacci price. Every water, every harvest, every
collection, every step across the board spends one.

That reframes the question from "what is profitable per tile" to "what is profitable per
action", and it is why the melon's ten-day watering schedule is expensive and why a
one-action-per-day fertilizer dividend is cheap.

In [ ]:
SEASON_TURNS = 24 * 30
print(f'season budget: {SEASON_TURNS} farmer actions')
print(f'hiring n hands for a day costs fib(1..n) = '
      f'{[1, 1, 2, 3, 5, 8, 13, 21]} → 8 hands for one day costs $54 '
      f'and buys {8 * 24} extra actions')
print(f'\nso an extra action is worth roughly ${54 / (8 * 24):.2f} at the margin — '
      f'anything earning more than that per action is worth doing')

Hired hands are absurdly cheap relative to what an action earns. Eight hands for a
day cost $54 and hand you 192 extra actions; a single melon harvest is worth more than
that. **If your agent is not hiring aggressively every day, that is probably the largest
single number on the table** — larger than any crop choice in section 2.

(Caveat worth checking in the engine: hands spawn adjacent to the shed, and the first hire
each day spawns onto a tile that is locked until you buy the NE quadrant. Locked tiles are
passable, so it costs a move, not the hand.)

## What to do with this

1. **Ignore base prices.** Rank by `revenue(T)`, not by the price column.
2. **Melon for the money, wheat and egg for the floor.** Wheat and egg are the only two
   goods with no crash — they are where a large surplus goes without being destroyed.
3. **Never plant a full field of strawberries.** Its ceiling is $3,847 and it hits the
   floor at 62 units.
4. **Collect fertilizer from every animal, every day.** It is free, it is the
   second-largest revenue pool in the game, and for sheep it dwarfs the wool.
5. **Feed every other day, not every day.** Half the actions, same base yield.
6. **Hire hands.** An action costs about $0.28 and earns far more.

All of the above is arithmetic on the published rules, not results from the simulator. If
you run it and something disagrees, say so in the comments — I would rather the numbers
be right than be mine.